## This notebook
Calculates the percentage of segdup in the genome 

In [ ]:
import numpy as np
from scipy.stats import fisher_exact, binomtest, hypergeom
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import splprep, splev
import pyranges as pr

In [ ]:
# New-annotation GFF (final AGAT-normalized annotation; re-run with this path)
GFF_PATH = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/code/command-line-script/annotation-merging/output/hifiasm-041425-denovoEnhanced_peaks2utr_sorted.agat.gff3"
gff = pd.read_csv(GFF_PATH, sep='\t', comment="#", header=None,
                  names=["Sequence","source","type","Gene Start","Gene End","score","Strand","phase","attributes"])

gff = gff[gff["type"]=="gene"].copy()
gff['gene'] = gff['attributes'].str.extract(r'ID=gene-([^;]+)', expand=False).str.upper()

In [ ]:
# Segmental-duplication BEDPE (genome-level, unchanged)
BEDPE_PATH = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj/code/command-line-script/genome-annotation/biser/hifiasm-041425/segdup_output_duplicateLinkRemoved.bedpe"
dup = pd.read_csv(BEDPE_PATH, sep="\t", header=None,
                  names=["chr1","start1","end1","chr2","start2","end2","reference","score","strand1","strand2","max_len","aln_len","cigar","optional"])

In [ ]:
def merge_alignments(df, chrom_col='chr1', start_col='start1', end_col='end1', type_col='aln_type'):
    """
    Merge overlapping or adjacent alignments of the same type on the same chromosome
    Returns a DataFrame with merged intervals and their total lengths
    """
    merged_data = []
    
    # Group by chromosome and alignment type
    for (chrom, aln_type), group in df.groupby([chrom_col, type_col]):
        # Sort by start position
        sorted_group = group.sort_values(start_col)
        
        merged_intervals = []
        current_interval = None
        
        for _, row in sorted_group.iterrows():
            start, end = row[start_col], row[end_col]
            
            if current_interval is None:
                # Start first interval
                current_interval = [start, end]
            elif start <= current_interval[1]:
                # Overlapping or adjacent - merge
                current_interval[1] = max(current_interval[1], end)
            else:
                # Non-overlapping - save current and start new
                merged_intervals.append({
                    'chrom': chrom,
                    'aln_type': aln_type,
                    'start': current_interval[0],
                    'end': current_interval[1],
                    'length': current_interval[1] - current_interval[0]
                })
                current_interval = [start, end]
        
        # Don't forget the last interval
        if current_interval is not None:
            merged_intervals.append({
                'chrom': chrom,
                'aln_type': aln_type,
                'start': current_interval[0],
                'end': current_interval[1],
                'length': current_interval[1] - current_interval[0]
            })
        
        merged_data.extend(merged_intervals)
    
    return pd.DataFrame(merged_data)

In [ ]:
dup["aln_type"]="all"

In [ ]:
merged_dup = merge_alignments(dup)

In [ ]:

# # Create mock gene features DataFrame (BUSCO-like)
# all_features = pd.DataFrame({
#     "Busco id": ["gene1", "gene2", "gene3", "gene4"],
#     "Status": ["Complete"] * 4,
#     "Sequence": ["chr1", "chr1", "chr2", "chr2"],
#     "Gene_Start": [1000, 5000, 10000, 12500],
#     "Gene_End": [2000, 6000, 11000, 12600],
#     "Strand": ["+", "-", "+","-"],
#     "Score": [100, 200, 150, 100],
#     "Length": [1000, 1000, 1000, 10],
#     "OrthoDB url": ["url1", "url2", "url3", "url4"],
#     "Description": ["desc1", "desc2", "desc3", "desc4"]
# })

# # Create mock segmental duplication DataFrame (non-overlapping)
# dup = pd.DataFrame({
#     "reference": ["ref1", "ref2", "ref3"],
#     "chrom": ["chr1", "chr1", "chr2"],
#     "start": [3000, 7000, 12000],
#     "end": [4000, 8000, 13000],
#     "core_id": ["dup1", "dup2", "dup3"],
#     "len": [1000, 1000, 1000],
#     "score": [900, 800, 850],
#     "strand": ["+", "-", "+"],
#     "type": [None, None, None]
# })


all_features=gff.copy()
# Format for PyRanges
all_features["Start"] = all_features["Gene Start"] - 1  # Convert to 0-based start
all_features["End"] = all_features["Gene End"]
all_features = all_features.rename(columns={"Sequence": "Chromosome"})

dup["Start"] = dup["start"] - 1  # Convert to 0-based start
dup["End"] = dup["end"]
dup = dup.rename(columns={"chrom": "Chromosome", "strand": "Strand"})

# Create PyRanges objects
busco_pr = pr.PyRanges(all_features[["Chromosome", "Start", "End", "Strand"]])
dup_pr = pr.PyRanges(dup[["Chromosome", "Start", "End", "Strand"]])

# Perform overlap (returns only genes that overlap duplications)
overlaps = busco_pr.join(dup_pr)

# Extract coordinates of overlapping genes (as tuples for fast lookup)
if not overlaps.df.empty:
    overlapping_genes = set(zip(
        overlaps.df["Chromosome"],
        overlaps.df["Start"],
        overlaps.df["End"],
        overlaps.df["Strand"]
    ))
else:
    overlapping_genes = set()

# Add overlap indicator column
all_features["overlaps_dup"] = all_features.apply(
    lambda row: (row["Chromosome"], row["Start"], row["End"], row["Strand"]) in overlapping_genes,
    axis=1
)

# Show results
print("Genes overlapping segmental duplications:")
print(all_features[[ "Chromosome", "Start", "End", "overlaps_dup"]])

In [ ]:
all_features.to_csv("./hifiasm_gene_segDup_overlapInfo_092525.tsv", sep="\t")